[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-05-serialization.ipynb#scrollTo=aa11bb22)

---
# Day 5 · Serialization — model_dump, model_dump_json, and JSON Schema
**certified-journeys / pydantic-certified** · Day 5 · Output and Integration

> **Goal for today:** Control exactly what your models produce when serialized — using `model_dump` flags, `model_dump_json` for high-performance output, `@computed_field` for derived data, `@field_serializer` for custom formatting, and `model_json_schema` for API documentation.


In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings


## Step 1 · `model_dump()` — The Workhorse Serializer

`model_dump()` converts a Pydantic model to a Python dictionary. Several flags control what you get:

| Flag | Effect |
|---|---|
| `exclude_unset=True` | Only fields that were explicitly set (not defaults) |
| `exclude_defaults=True` | Exclude fields whose value equals the default |
| `exclude_none=True` | Exclude fields with `None` value |
| `include={'field1', 'field2'}` | Whitelist — only these fields |
| `exclude={'field3'}` | Blacklist — all except these fields |
| `mode='json'` | Convert non-JSON-native types (datetime → ISO string, UUID → str) |

`exclude_unset=True` is especially important for **PATCH APIs**: send only what the client changed, not the full model with defaults.

> **Reading:** [Pydantic Serialization](https://docs.pydantic.dev/latest/concepts/serialization/)


In [ ]:
from datetime import datetime, timezone
from typing import Optional
from pydantic import BaseModel


class UserProfile(BaseModel):
    id: int
    username: str
    email: Optional[str] = None
    bio: str = ""
    is_active: bool = True
    created_at: datetime = datetime(2024, 1, 1, tzinfo=timezone.utc)


# Full model — all fields
full = UserProfile(id=1, username="alice", email="alice@example.com")
print("=== model_dump() — all fields ===")
print(full.model_dump())

# PATCH scenario: only send what the client actually provided
patch = UserProfile(id=1, username="alice", bio="Python engineer")
print("\n=== exclude_unset=True (PATCH scenario) ===")
print(patch.model_dump(exclude_unset=True))
# → {'id': 1, 'username': 'alice', 'bio': 'Python engineer'}
# email, is_active, created_at are absent because client didn't send them

# Exclude fields that equal their default value
print("\n=== exclude_defaults=True ===")
print(full.model_dump(exclude_defaults=True))

# Exclude None values
no_email = UserProfile(id=2, username="bob")
print("\n=== exclude_none=True ===")
print(no_email.model_dump(exclude_none=True))

# mode='json' → datetime becomes ISO string instead of datetime object
print("\n=== mode='json' — datetime as ISO string ===")
d = full.model_dump(mode="json")
print(d)
print("Type of created_at:", type(d["created_at"]))  # <class 'str'>


### What just happened?
- **`exclude_unset=True`** is the correct tool for PATCH endpoints — it returns only what was explicitly provided, preserving the difference between "field was not sent" and "field was set to its default".
- **`mode='json'`** is a safe way to get a JSON-serializable dict without calling `json.dumps` — `datetime`, `UUID`, `Decimal`, `Path` all become strings automatically.
- **`exclude_defaults=True`** and **`exclude_none=True`** reduce payload size and noise in logs.
- The difference between `exclude_unset` and `exclude_defaults`: a field set explicitly to its default value appears in `exclude_defaults=True` output but NOT in `exclude_unset=True` if it wasn't in the input.


## Step 2 · `model_dump_json()` — Direct JSON Output

`model_dump_json()` produces a **JSON bytestring** by calling Pydantic's Rust-based serializer directly — it never builds an intermediate Python dict.

```python
json_bytes: str = model.model_dump_json()
```

Performance comparison:

| Method | Path | Typical speedup |
|---|---|---|
| `model.model_dump() + json.dumps()` | Python dict → Python JSON encoder | baseline |
| `model.model_dump_json()` | Rust serializer directly | **2–4× faster** |

All the same flags work: `exclude_unset`, `exclude_none`, `include`, `exclude`.

> **Reading:** [model_dump_json docs](https://docs.pydantic.dev/latest/concepts/serialization/#modelmodel_dump_json)


In [ ]:
import json
import time
from datetime import datetime, timezone
from uuid import UUID, uuid4
from pydantic import BaseModel


class Event(BaseModel):
    id: UUID
    name: str
    timestamp: datetime
    payload: dict


event = Event(
    id=uuid4(),
    name="user.signup",
    timestamp=datetime.now(timezone.utc),
    payload={"user_id": 42, "plan": "pro"},
)

# model_dump_json returns a str (JSON string)
json_str = event.model_dump_json()
print("model_dump_json():")
print(json_str)
print("Type:", type(json_str))  # <class 'str'>

# Same flags as model_dump
print("\nWith indent:")
print(event.model_dump_json(indent=2))

# Benchmark: model_dump_json vs model_dump + json.dumps
N = 50_000

start = time.perf_counter()
for _ in range(N):
    json.dumps(event.model_dump(mode="json"))
t_two_step = (time.perf_counter() - start) * 1000

start = time.perf_counter()
for _ in range(N):
    event.model_dump_json()
t_direct = (time.perf_counter() - start) * 1000

print(f"\nBenchmark ({N:,} iterations):")
print(f"  model_dump(mode='json') + json.dumps : {t_two_step:.1f} ms")
print(f"  model_dump_json()                    : {t_direct:.1f} ms")
print(f"  Speedup: {t_two_step / t_direct:.1f}x")


### What just happened?
- `model_dump_json()` returns a `str` — the raw JSON — not a `bytes` object and not a dict.
- **UUID** and **datetime** are serialized to strings automatically (no custom encoder needed).
- The performance advantage is real: the Rust serializer avoids building a temporary Python dict, which matters in high-throughput APIs.
- `indent=2` works just like `json.dumps(indent=2)` — useful for debugging and log output.


## Step 3 · `@computed_field` — Derived Properties in Serialized Output

`@computed_field` exposes a Python `@property` as a **first-class field** in `model_dump()` and `model_dump_json()` output.

Without `@computed_field`: the property exists on the Python object but is **silently omitted** from serialization.  
With `@computed_field`: the property appears in dumps and JSON schema automatically.

Use it for:
- Derived display values (`full_name = first_name + last_name`)
- Computed aggregates (`total_price = quantity * unit_price`)
- Cached expensive computations

> **Reading:** [Pydantic Computed Fields](https://docs.pydantic.dev/latest/concepts/serialization/#computed-fields)


In [ ]:
from functools import cached_property
from pydantic import BaseModel, computed_field


class OrderItem(BaseModel):
    product_name: str
    quantity: int
    unit_price: float  # in dollars

    @computed_field
    @property
    def total_price(self) -> float:
        """Derived from quantity × unit_price."""
        return round(self.quantity * self.unit_price, 2)


class Order(BaseModel):
    order_id: str
    items: list[OrderItem]
    tax_rate: float = 0.08

    @computed_field
    @property
    def subtotal(self) -> float:
        return round(sum(item.total_price for item in self.items), 2)

    @computed_field
    @property
    def tax_amount(self) -> float:
        return round(self.subtotal * self.tax_rate, 2)

    @computed_field
    @property
    def grand_total(self) -> float:
        return round(self.subtotal + self.tax_amount, 2)


order = Order(
    order_id="ORD-001",
    items=[
        OrderItem(product_name="Widget", quantity=3, unit_price=9.99),
        OrderItem(product_name="Gadget", quantity=1, unit_price=49.95),
    ],
)

# Computed fields appear in model_dump output
print("model_dump():")
import json
print(json.dumps(order.model_dump(), indent=2))

# Also in JSON output
print("\nmodel_dump_json (excerpt):")
print(order.model_dump_json(indent=2))


### What just happened?
- `@computed_field` requires `@property` (or `@cached_property`) underneath it — the decorator stack order is `@computed_field` on top, `@property` below.
- `total_price`, `subtotal`, `tax_amount`, and `grand_total` all appear in `model_dump()` output — they're treated like stored fields.
- **`@cached_property`** (from `functools`) works too — use it when the computation is expensive and the fields it depends on don't change.
- Computed fields are **read-only** — you cannot set them during model construction.


## Step 4 · `@field_serializer` — Customize Field Output

`@field_serializer` lets you control how a **specific field** is serialized — without changing how it's stored or validated.

```python
@field_serializer('my_field')
def serialize_my_field(self, value: T, info: FieldSerializationInfo) -> Any:
    return transformed_value
```

Use cases:
- Format a `Decimal` as a string with fixed precision
- Mask sensitive fields (`password` → `"***"`)
- Convert an enum to a human-readable label
- Serialize a `set` as a sorted list (JSON has no set type)

> **Reading:** [Pydantic field_serializer](https://docs.pydantic.dev/latest/concepts/serialization/#field-serializer)


In [ ]:
from decimal import Decimal
from enum import Enum
from typing import Set
from pydantic import BaseModel, field_serializer


class Status(str, Enum):
    ACTIVE = "active"
    INACTIVE = "inactive"
    PENDING = "pending"


class Account(BaseModel):
    username: str
    password: str          # stored as-is, serialized as masked
    balance: Decimal       # serialized with 2 decimal places
    status: Status         # serialized as human-readable label
    tags: Set[str]         # serialized as sorted list

    @field_serializer("password")
    def mask_password(self, value: str) -> str:
        """Never expose passwords in serialized output."""
        return "***"

    @field_serializer("balance")
    def format_balance(self, value: Decimal) -> str:
        """Always output balance with 2 decimal places as string."""
        return f"{value:.2f}"

    @field_serializer("status")
    def humanise_status(self, value: Status) -> str:
        """Return a title-cased label instead of the raw enum value."""
        labels = {Status.ACTIVE: "Active", Status.INACTIVE: "Inactive", Status.PENDING: "Awaiting Review"}
        return labels[value]

    @field_serializer("tags")
    def sorted_tags(self, value: Set[str]) -> list[str]:
        """JSON has no set type; return a deterministic sorted list."""
        return sorted(value)


acct = Account(
    username="alice",
    password="super_secret_123",
    balance=Decimal("1234.5"),
    status=Status.ACTIVE,
    tags={"premium", "verified", "beta"},
)

print("model_dump():")
import json
print(json.dumps(acct.model_dump(), indent=2))

print("\nPassword is still accessible on the model object:")
print(acct.password)  # → 'super_secret_123'


### What just happened?
- `@field_serializer` runs only during serialization — it does **not** change what's stored in the model instance (`acct.password` is still the real password).
- Masking passwords in `field_serializer` is a clean defence-in-depth pattern: even if you accidentally serialize the model, the secret won't leak.
- Serializing `Decimal` as a fixed-precision string prevents floating-point representation issues in downstream JSON consumers.
- The `tags: Set[str]` → `sorted(value)` pattern ensures **deterministic output** — critical for testing and caching.


## Step 5 · `model_json_schema()` — API Documentation and Validation

Pydantic generates a **JSON Schema** from any model. This is what FastAPI uses to power its `/docs` (Swagger UI) automatically.

```python
schema = MyModel.model_json_schema()
```

You can enrich the schema with:
- `Field(description="...", examples=[...])` — human-readable docs
- `model_config = ConfigDict(json_schema_extra={...})` — merge extra properties
- `title` parameter on `Field` — override the auto-generated property name

> **Reading:** [Pydantic JSON Schema](https://docs.pydantic.dev/latest/concepts/json_schema/)


In [ ]:
import json
from typing import Optional
from pydantic import BaseModel, Field


class CreateUserRequest(BaseModel):
    """Request body for creating a new user account."""

    username: str = Field(
        min_length=3,
        max_length=32,
        pattern=r"^[a-z0-9_]+$",
        description="Lowercase alphanumeric username, underscores allowed",
        examples=["alice", "bob_smith"],
    )
    email: str = Field(
        description="Valid email address for account notifications",
        examples=["alice@example.com"],
    )
    age: Optional[int] = Field(
        default=None,
        ge=13,
        le=120,
        description="Age in years; must be 13 or older",
    )
    referral_code: Optional[str] = Field(
        default=None,
        max_length=16,
        description="Optional referral code from an existing user",
    )


schema = CreateUserRequest.model_json_schema()

print("JSON Schema:")
print(json.dumps(schema, indent=2))

print("\n--- Key observations ---")
print("Title:      ", schema["title"])
print("Description:", schema.get("description", "none"))
print("Required:   ", schema.get("required", []))
print("Properties: ", list(schema["properties"].keys()))
print("username constraints:")
import pprint
pprint.pprint(schema["properties"]["username"])


### What just happened?
- The `"description"` and `"examples"` from `Field(...)` appear verbatim in the schema — FastAPI exposes these in Swagger UI.
- `min_length`, `max_length`, `pattern`, `ge`, `le` all map to standard JSON Schema keywords (`minLength`, `maxLength`, `pattern`, `minimum`, `maximum`).
- `Optional[int]` fields appear in `properties` but not in `required` — the schema accurately reflects optionality.
- The docstring on the class becomes the schema's `"description"` property — document your models and get API docs for free.


In [ ]:
# Challenge: Serialization control
#
# 1. Create a model `Report` with:
#      title: str
#      created_at: datetime  (use datetime.now(timezone.utc) as default_factory)
#      rows: list[dict]      (raw data rows)
#      author_email: str     (must be excluded from all serialization output)
#
# 2. Add a @computed_field `row_count` that returns len(self.rows).
#
# 3. Add a @field_serializer for `created_at` that outputs it as
#    'YYYY-MM-DD HH:MM' (no seconds, no timezone offset) — e.g. '2024-07-15 09:30'.
#
# 4. Demonstrate:
#    a) model_dump_json() — confirm created_at format and row_count present
#    b) model_dump(exclude={'author_email'}) — confirm email is absent
#    c) model_json_schema() — confirm row_count appears in properties

from datetime import datetime, timezone
from pydantic import BaseModel, Field, computed_field, field_serializer

# Your solution here
# class Report(BaseModel):
#     ...


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `model_dump(exclude_unset=True)` | PATCH-safe: only returns explicitly-set fields |
| `model_dump(mode='json')` | Converts datetime/UUID/Decimal to JSON-safe Python types |
| `model_dump_json()` | Rust serializer — 2–4× faster than `model_dump() + json.dumps()` |
| `@computed_field` | Exposes `@property` in serialization and JSON schema output |
| `@field_serializer` | Customizes one field's output without changing stored value |
| `model_json_schema()` | Generates JSON Schema; powered by `Field(description=..., examples=...)` |

> **Tip:** `model_dump_json()` is not just a convenience wrapper — it bypasses Python dict construction and calls the Rust serializer directly. For high-throughput APIs, this can be 2–4× faster than `model.model_dump() + json.dumps()`.

---
## What's next
**Day 6** → Settings management with `pydantic-settings`: load config from `.env` files and environment variables with full type validation.

Mark Day 5 complete in your [tracker](../index.html).
